Write a python program to find the second-largest number in a list without using sort()

In [0]:
numbers = [10,25,5,40,30,40,]

In [0]:
largest = second = float('-inf')

for num in numbers:
    if num > largest:
        largest, second = num, largest
    elif second < num < largest:
        second = num

print(second if second != float('-inf') else None)

In [0]:
# Find the second-largest number in a list without using sort().
# Strategy: track the two biggest values seen so far in a single pass.

# Start both trackers at negative infinity so any real number beats them.
largest = second = float('-inf')

for num in numbers:
    if num > largest:
        # New maximum found: the old largest gets demoted to second-largest.
        largest, second = num, largest
    elif second < num < largest:
        # num is not the largest, but it beats the current second-largest.
        # (num == largest is skipped, so duplicates of the max don't count
        # as the second-largest.)
        second = num

# If second was never updated, the list had fewer than 2 distinct values.
print(second if second != float('-inf') else None)

In [0]:
def second_largest(nums):
    """Return the second-largest distinct value in nums, or None if it doesn't exist.

    Single pass, O(n) time / O(1) space. Duplicates of the maximum don't
    count as second-largest ([40, 40] -> None).
    """
    largest = second = float('-inf')

    for num in nums:
        if num > largest:
            largest, second = num, largest          # new max: old max demoted
        elif second < num < largest:
            second = num                            # beats second, isn't a dup of max

    return second if second != float('-inf') else None


# Tests -- interviewers love that you thought about edge cases:
print(second_largest([10, 25, 5, 40, 30, 40]))  # 30  (duplicate 40s don't win)
print(second_largest([40, 40]))                # None (no 2nd distinct value)
print(second_largest([7]))                    # None (too short)
print(second_largest([-5, -2, -9]))           # -5 (works with negatives)
print(second_largest([]))                     # None (empty)

In [0]:
def second_largest(nums):
    """Return the second-largest distinct value in nums, or None if it doesn't exist.

    Same single-pass O(n) / O(1) approach, but uses None as the sentinel
    instead of float('-inf') -- no floats involved at all.
    """
    largest = second = None

    for num in nums:
        if largest is None or num > largest:
            largest, second = num, largest      # new max: old max demoted
        elif num != largest and (second is None or num > second):
            second = num                        # distinct from max, beats second

    return second


# Same test suite -- results must match the float('-inf') version:
print(second_largest([10, 25, 5, 40, 30, 40]))  # 30
print(second_largest([40, 40]))                # None
print(second_largest([7]))                    # None
print(second_largest([-5, -2, -9]))           # -5
print(second_largest([]))                     # None

In [0]:
def second_largest(numbers):
    largest=second=None
    for num in numbers:
        if largest is None or num > largest:
            largest,second =num,largest
        elif num != largest and (second is None or num > second):
            second = num
    return second

In [0]:
second_largest(numbers)

In [0]:
def second_largest(numbers):
    largest=second=None
    for num in numbers:
        if largest is None or num > largest:
            largest,second = num,largest
        elif num != largest and (second is None or num > second):
            second = num
    return second
second_largest(numbers)

In [0]:
def second_largest(numbers):
    largest=second=None
    for num in numbers:
        if largest is None or num > largest:
            largest,second = num,largest
        elif num!=largest and (second is None or num > second):
            second = num
    return second
numbers = [10,25,5,40,30,40]
second_largest(numbers)

Write a python program to count the frequency of each word in a string.

In [0]:
text ='python spark python sql spark python'

In [0]:
d={}
for word in text.split(" "):
    if d.get(word):
        d[word]+=1
    else:
        d[word]=1
print(d)

Write a python function to find common elements between two lists without duplicates

In [0]:
a=[1,2,3,4,5]
b=[3,4,5,6,7]

common=[]
for x in a:
    if x in b and x not in common:   # in b => common; not in common => skip duplicates
        common.append(x)
print(common)   # [3, 4, 5]

In [0]:
common=[]
for x in a:
    for y in b:
        if x==y:
            common.append(y)
print(common)

Write a python program to flatten nested list

In [0]:
data = [[1,2],[3,4],[5,6]]

flattened = []
for sublist in data:
    for item in sublist:
        flattened.append(item)
print(flattened)   # [1, 2, 3, 4, 5, 6]


In [0]:
flattened1=[]
for sublist in data:
    for item in sublist:
        flattened1.append(item)

print(flattened1)



In [0]:
data = [[1,2],[3,4],[5,6]]

flattened = [item for sublist in data for item in sublist]
print(flattened)   #

In [0]:
# Processing a 10M-row CSV without loading it all into memory
# -----------------------------------------------------------
# Key ideas:
#   1. Stream the file in small chunks (pandas chunksize or csv module) --
#      only `chunk_size` rows live in RAM at any moment.
#   2. Aggregate/transform each chunk, then combine the partial results.
#   3. For truly heavy lifting, Spark (Databricks) reads CSV lazily and
#      distributed by default -- see the last section.

import pandas as pd

CSV_PATH = "/path/to/big_file.csv"
chunk_size = 100_000  # ~100k rows per chunk; tune based on row width and RAM


# --- Approach 1: pandas chunked reading (streaming aggregation) -------------
def process_with_pandas(path):
    """Example: compute total row count and sum of the 'amount' column."""
    total_rows = 0
    amount_sum = 0.0

    for chunk in pd.read_csv(path, chunksize=chunk_size):
        total_rows += len(chunk)
        amount_sum += chunk["amount"].sum()   # do per-chunk work here

    return total_rows, amount_sum


# --- Approach 2: plain csv module (lowest memory, no pandas dependency) -----
import csv

def process_with_csv_module(path):
    """Row-by-row streaming: constant memory, works for any file size."""
    with open(path, newline="") as f:
        reader = csv.DictReader(f)           # streams one row at a time
        for row in reader:
            yield transform(row)             # generator keeps memory flat


def transform(row):
    # example transformation -- replace with real logic
    return {k: v for k, v in row.items() if k != "unwanted_column"}


# --- Approach 3: PySpark (best in Databricks -- distributed & lazy) --------
# Spark never loads the whole file into one machine's memory; it splits the
# CSV into partitions and processes them in parallel across the cluster.

# df = spark.read.csv(CSV_PATH, header=True, inferSchema=True)
# result = df.groupBy("category").agg(sum("amount"), count("*"))
# result.show()   # only executes now, and only the aggregate is collected


# --- Memory-saving extras worth knowing -------------------------------------
#   * dtype=strings_as_categories / dtype=... in read_csv: smaller footprint
#   * usecols=[...]: read only the columns you need
#   * engine="pyarrow" with pyarrow.csv: very fast streaming reader

# Quick demo (uncomment to run):
# rows, total = process_with_pandas(CSV_PATH)
# print(f"rows={rows:,}, amount_sum={total:,.2f}")
